In [7]:
import csv, requests, tqdm

In [2]:
API_KEY = '36e7b6c5c096fa274fe7b923ed298a463f30007b6158a56a8898d7e616ff066330e7eba6b564dfbf0fa574c6cf83d01c257f29f5c875af7ad1951f88cc4d2d6a42e57379146824d6aafbe32ff89792ffa49af18de5c8e58dbd9d4a9546ece0742a62a38b4eeea28b6fc3c331dc28336ef04fb1106e7c6a9d4fd8619744527b51'

API_ROOT = 'http://localhost:1337/'

In [3]:
filters = [{
    'uuid': {
        '$eq': 'd0a1542b-e085-4c2d-99ab-52427a1d26e1'
    }
}]

# Helper function to make writing filters a bit less error-prone
# Target data structure: GET /api/users?filters[username][$eq]=John

def get_filters(filters):
    filter_list = []
    for f_dict in filters:
        for field, filtercomponent in f_dict.items():
            for filtertype, value in filtercomponent.items():
                filterstring = f'filters[{field}][{filtertype}]={value}'
                filter_list.append(filterstring)
    return filter_list

## Upload Data

In [228]:
def upload_data(filepath, endpoint, api_key, id_fields, exclude_fields=None):
    """
    Upload the data from a given file. Doesn't work on Makers because the 'id' value is invalid for strapi as it clashes with the internal ID value - needs to be converted
    to be 'maker_id'
    param str filepath: the path to the file containing the data in csv format
    param str endpoint: the url of the api endpoint
    param str api_key: the key for the api
    param list str id_fields: fieldnames of ids that need to be cast to INTs
    param list str exclude_fields: fieldnames to exclude from the CSV (if they aren't in the data model, for eg)
    """
    data = []
    
    with open(filepath, encoding='utf-8-sig') as file:
        reader = csv.DictReader(file)
        for row in reader:
            data.append(row)
        file.close()
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f'Bearer {api_key}',
    }
    
    for row in tqdm.tqdm(data):
        record = dict(row)
        
        for fieldname in id_fields:
            try:
                record[fieldname] = int(record[fieldname])
            except:
                record[fieldname] = None
                
        if exclude_fields:
            for item in exclude_fields:
                del record[item]
        
        payload = {
            'data': record
        }
        
        r = requests.post(endpoint, json=payload, headers=headers)
        if r.ok:
            continue
        else:
            print(r.status_code, r.text)
        

In [ ]:
# Makers

filepath = './data/csv/maker.csv'
endpoint = API_ROOT + 'api/makers/'
id_fields = ['maker_id']
upload_data(filepath=filepath, endpoint=endpoint, api_key=API_KEY, id_fields=id_fields)

In [33]:
# Addresses

filepath = './data/csv/address.csv'
endpoint = API_ROOT + 'api/addresses/'
id_fields = ['address_id', 'maker_id', 'town_location_id']
upload_data(filepath=filepath, endpoint=endpoint, api_key=API_KEY, id_fields=id_fields)

In [151]:
# Town Locations

filepath = './data/csv/town-location.csv'
endpoint = API_ROOT + 'api/town-locations/'
id_fields = ['town_location_id']
upload_data(filepath=filepath, endpoint=endpoint, api_key=API_KEY, id_fields=id_fields)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 618/618 [00:03<00:00, 193.16it/s]


In [225]:
# Guilds

filepath = './data/csv/guild.csv'
endpoint = API_ROOT + 'api/guilds/'
id_fields = ['guild_id']
upload_data(filepath=filepath, endpoint=endpoint, api_key=API_KEY, id_fields=id_fields)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 58/58 [00:00<00:00, 146.53it/s]


In [221]:
# Memberships

filepath = './data/csv/membership.csv'
endpoint = API_ROOT + 'api/memberships/'
id_fields = ['membership_id', 'guild_id', 'maker_id']
exclude_fields = ['entry', 'entry_date_1', 'entry_date_2', 'uncertain', 'misc_codes', 'entry_date_1_certainty', 'entry_date_2_certainty']

upload_data(filepath=filepath, endpoint=endpoint, api_key=API_KEY, id_fields=id_fields, exclude_fields=exclude_fields)


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1841/1841 [00:07<00:00, 259.49it/s]


In [ ]:
# Relations

filepath = './data/csv/relation.csv'
endpoint = API_ROOT + 'api/relations/'
id_fields = ['maker_id', 'relation_code', 'relation_id', 'relation_type_id']
exclude_fields = ['relation_type_meta_id']

upload_data(filepath=filepath, endpoint=endpoint, api_key=API_KEY, id_fields=id_fields, exclude_fields=exclude_fields)

In [233]:
# Disambiguated Relations

filepath = './data/csv/disambiguated-relation.csv'
endpoint = API_ROOT + 'api/disambiguated-relations/'
id_fields = ['target_maker_id', 'relation_id']
exclude_fields = ['is_auto_disambiguation']

upload_data(filepath=filepath, endpoint=endpoint, api_key=API_KEY, id_fields=id_fields, exclude_fields=exclude_fields)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10252/10252 [00:34<00:00, 301.09it/s]


In [265]:
# Instruments known

filepath = './data/csv/instrument-known.csv'
endpoint = API_ROOT + 'api/instruments-known/'
id_fields = ['maker_id', 'inst_code', 'id']
exclude_fields = ['id']

upload_data(filepath=filepath, endpoint=endpoint, api_key=API_KEY, id_fields=id_fields, exclude_fields=exclude_fields)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6208/6208 [00:20<00:00, 300.94it/s]


In [263]:
# Instruments Advertised

filepath = './data/csv/instrument-advertised.csv'
endpoint = API_ROOT + 'api/instruments-advertised/'
id_fields = ['maker_id', 'inst_code', 'id']
exclude_fields = ['id']

upload_data(filepath=filepath, endpoint=endpoint, api_key=API_KEY, id_fields=id_fields, exclude_fields=exclude_fields)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9156/9156 [00:28<00:00, 322.39it/s]


In [250]:
def connect_records(source_document_id, target, connection_fieldname, api_key):
    """
    Connect one record to another
    param dict source_document_id: the source record documentId
    param dict target: {'endpoint': 'http://localhost:1337/api/endpoint/', 'id': 1}
    param str connection_fieldname: the fieldname that holds the relation value
    """
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f'Bearer {api_key}',
    }
    
    ## Get the target record
            
    target_filters = [{
        target['id_fieldname']: {
            '$eq': target['id']
        }
    }]
    
    target_request = requests.get(target['endpoint'] + f'?{get_filters(target_filters)[0]}')
    
    target_document_id = None
    
    if target_request.ok:
        target_data = target_request.json()
        if (len(target_data['data'])):
           target_document_id = target_data['data'][0]['documentId']
    
    # If you have both source and target, connect them using the correct fieldname
    
    if (source_document_id and target_document_id):
        
        payload = {
            'data': {
                connection_fieldname: {
                    'connect': [source_document_id]
                }
            }
        }
        
        url = target['endpoint'] + f'{target_document_id}'

        conn_r = requests.put(url, json=payload, headers=headers)
        
        if conn_r.ok:
             return True
        else:
            return False
        
    
    else:
        return False
    
    

In [268]:
def fetch_records_and_connect(source_endpoint, target_endpoint, source_id_fieldname, target_id_fieldname, connection_fieldname, pages_to_process=None):
    """
    Iterate over paginated records and connect them
    param str source_endpoint: the url of the source endpoint
    param str target_endpoint: the url of the target endpoint
    param str source_id_fieldname: the SIMON ID of the source relation to connect
    param str target_id_fieldname: the SIMON ID of the target relation to connect
    param str connection_fieldname: the fieldname of the relation connection for Strapi
    
    """
    
    ok = []
    errors = []
    
    page = 1
    page_count = None
    
    # get the total page count
    r = requests.get(source_endpoint)
    
    if r.ok:
        r_data = r.json()
        page_count = r_data['meta']['pagination']['pageCount']
        
    if pages_to_process:
        total_pages = pages_to_process
    else:
        total_pages = page_count
    
    while (page <= total_pages):
        
        source_data = None
        
        url = source_endpoint + f'?pagination[page]={page}&pagination[pageSize]=25'
        
        r = requests.get(url)

        if r.ok:
            source_data_json = r.json()
            source_data = source_data_json['data']
    
        if (source_data):
            for row in source_data:
                target = None

                if row[source_id_fieldname]:
                    target = {
                        'endpoint': target_endpoint,
                        'id': row[source_id_fieldname],
                        'id_fieldname': target_id_fieldname
                    }

                if target:
                    connected = connect_records(source_document_id=row['documentId'], target=target, connection_fieldname=connection_fieldname, api_key=API_KEY)
                    if connected:
                        ok.append(row['documentId'])
                    else:
                        errors.append(row['documentId'])
        print(f'processed page {page} of {total_pages}')
        
        page += 1
        
    
    return {'errors': errors, 'ok': ok}

In [ ]:
# Connect addresses and makers

source_endpoint = API_ROOT + 'api/addresses/'
target_endpoint = API_ROOT + 'api/makers/'
id_fieldname = 'maker_id'
connection_fieldname = 'addresses'

conns = fetch_records_and_connect(source_endpoint=source_endpoint, target_endpoint=target_endpoint, id_fieldname=id_fieldname, connection_fieldname=connection_fieldname)

In [185]:
len(conns['ok'])

2400

In [ ]:
# Connect addresses and town locations

source_endpoint = API_ROOT + 'api/addresses/'
target_endpoint = API_ROOT + 'api/town-locations/'
id_fieldname = 'town_location_id'
connection_fieldname = 'addresses'

conns = fetch_records_and_connect(source_endpoint=source_endpoint, target_endpoint=target_endpoint, id_fieldname=id_fieldname, connection_fieldname=connection_fieldname)

In [ ]:
# Connect memberships and makers

source_endpoint = API_ROOT + 'api/memberships/'
target_endpoint = API_ROOT + 'api/makers/'
id_fieldname = 'maker_id'
connection_fieldname = 'memberships'

conns = fetch_records_and_connect(source_endpoint=source_endpoint, target_endpoint=target_endpoint, id_fieldname=id_fieldname, connection_fieldname=connection_fieldname)

In [ ]:
# Connect memberships and guilds

source_endpoint = API_ROOT + 'api/memberships/'
target_endpoint = API_ROOT + 'api/guilds/'
id_fieldname = 'guild_id'
connection_fieldname = 'memberships'

conns = fetch_records_and_connect(source_endpoint=source_endpoint, target_endpoint=target_endpoint, id_fieldname=id_fieldname, connection_fieldname=connection_fieldname)

In [ ]:
# Connect makers and relations

source_endpoint = API_ROOT + 'api/relations/'
target_endpoint = API_ROOT + 'api/makers/'
id_fieldname = 'maker_id'
connection_fieldname = 'relations'

conns = fetch_records_and_connect(source_endpoint=source_endpoint, target_endpoint=target_endpoint, id_fieldname=id_fieldname, connection_fieldname=connection_fieldname)

In [ ]:
# Connect disambiguated relations and relations

source_endpoint = API_ROOT + 'api/disambiguated-relations/'
target_endpoint = API_ROOT + 'api/relations/'
id_fieldname = 'relation_id'
connection_fieldname = 'disambiguated_relation'

conns = fetch_records_and_connect(source_endpoint=source_endpoint, target_endpoint=target_endpoint, id_fieldname=id_fieldname, connection_fieldname=connection_fieldname)

In [ ]:
# Connect disambiguated relations and makers

source_endpoint = API_ROOT + 'api/disambiguated-relations/'
target_endpoint = API_ROOT + 'api/makers/'
source_id_fieldname = 'target_maker_id'
target_id_fieldname = 'maker_id'
connection_fieldname = 'disambiguated_relations'

conns = fetch_records_and_connect(source_endpoint=source_endpoint, target_endpoint=target_endpoint, source_id_fieldname=source_id_fieldname, target_id_fieldname=target_id_fieldname, connection_fieldname=connection_fieldname)

In [ ]:
# Connect instruments known and makers

source_endpoint = API_ROOT + 'api/instruments-known/'
target_endpoint = API_ROOT + 'api/makers/'
source_id_fieldname = 'maker_id'
target_id_fieldname = 'maker_id'
connection_fieldname = 'instruments_known'

conns = fetch_records_and_connect(source_endpoint=source_endpoint, target_endpoint=target_endpoint, source_id_fieldname=source_id_fieldname, target_id_fieldname=target_id_fieldname, connection_fieldname=connection_fieldname)

In [270]:
# Connect instruments advertised and makers

source_endpoint = API_ROOT + 'api/instruments-advertised/'
target_endpoint = API_ROOT + 'api/makers/'
source_id_fieldname = 'maker_id'
target_id_fieldname = 'maker_id'
connection_fieldname = 'instruments_advertised'

conns = fetch_records_and_connect(source_endpoint=source_endpoint, target_endpoint=target_endpoint, source_id_fieldname=source_id_fieldname, target_id_fieldname=target_id_fieldname, connection_fieldname=connection_fieldname)

processed page 1 of 100
processed page 2 of 100
processed page 3 of 100
processed page 4 of 100
processed page 5 of 100
processed page 6 of 100
processed page 7 of 100
processed page 8 of 100
processed page 9 of 100
processed page 10 of 100
processed page 11 of 100
processed page 12 of 100
processed page 13 of 100
processed page 14 of 100
processed page 15 of 100
processed page 16 of 100
processed page 17 of 100
processed page 18 of 100
processed page 19 of 100
processed page 20 of 100
processed page 21 of 100
processed page 22 of 100
processed page 23 of 100
processed page 24 of 100
processed page 25 of 100
processed page 26 of 100
processed page 27 of 100
processed page 28 of 100
processed page 29 of 100
processed page 30 of 100
processed page 31 of 100
processed page 32 of 100
processed page 33 of 100
processed page 34 of 100
processed page 35 of 100
processed page 36 of 100
processed page 37 of 100
processed page 38 of 100
processed page 39 of 100
processed page 40 of 100
processed